In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# 1. 데이터 가져오기

In [3]:
import pandas as pd
df = pd.read_csv('C:/ai_x/download/shareData/부동산/최종전국평당분양가격(concat결측치제외).csv',
                encoding='cp949')
df.head()

,지역명,평당분양가격,연도,월
0,서울,18189.0,2013,12
1,부산,8111.0,2013,12
2,대구,8080.0,2013,12
3,인천,10204.0,2013,12
4,광주,6098.0,2013,12


# 2. 지역명의 라벨 인코딩
- 지역명을 라벨인코딩한 지역명2
- 분석할 경우 원핫인코딩까지 할 것을 추천

In [11]:
loc2 = df['지역명']
#라벨 인코딩
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
labeled_data = le.fit_transform(loc2)
df['지역명2'] = labeled_data

,지역명,평당분양가격,연도,월,지역명2
0,서울,18189.0,2013,12,8
1,부산,8111.0,2013,12,7
2,대구,8080.0,2013,12,5
3,인천,10204.0,2013,12,11
4,광주,6098.0,2013,12,4
...,...,...,...,...,...
2171,전북,12058.2,2024,8,13
2172,전남,13120.8,2024,8,12
2173,경북,13827.0,2024,8,3
2174,경남,13252.8,2024,8,2


In [12]:
#원핫 인코딩
from tensorflow.keras.utils import to_categorical
one_hot_encoded_data = to_categorical(labeled_data)
one_hot_encoded_data

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.]], dtype=float32)

# 3. normalization 스케일 조정
- 입력변수(지역명2, 연도, 월)와 타겟변수(평당분양가격) 따로 스케일 조정(MinMaxScaler이용)
- 지역명 2n, 연도n, 월n, 평당분양가격n 추가

In [24]:
x_data = df[['지역명2', '연도', '월']].values
y_data = df[['평당분양가격']].values.reshape(-1,1)

from sklearn.preprocessing import MinMaxScaler
import numpy as np
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()
scaled_x_data = scaler_x.fit_transform(x_data)
scaled_y_data = scaler_y.fit_transform(y_data)
scaled_y_data
np.column_stack([scaled_x_data, scaled_y_data])
df[['지역명 2n', '연도n', '월n']] = scaled_x_data
df['평당분양가격n'] = scaled_y_data
df.head()

,지역명,평당분양가격,연도,월,지역명2,지역명 2n,연도n,월n,평당분양가격n
0,서울,0.328198,0.0,1.0,8,0.5000,0.0,1.0,0.328198
1,부산,0.065274,0.0,1.0,7,0.4375,0.0,1.0,0.065274
2,대구,0.064466,0.0,1.0,5,0.3125,0.0,1.0,0.064466
3,인천,0.119878,0.0,1.0,11,0.6875,0.0,1.0,0.119878
4,광주,0.012757,0.0,1.0,4,0.2500,0.0,1.0,0.012757


# 4. standarization 스케일 조정
- 입력변수와 타겟변수 따로 스케일 조정(StandardScaler이용)
- 지역명 2s, 연도s, 월s필드 추가

In [34]:
x_data = df[['지역명2', '연도', '월']].values
y_data = df[['평당분양가격']].values.reshape(-1,1)

from sklearn.preprocessing import StandardScaler
import numpy as np
scaler_x = StandardScaler()
scaler_y = StandardScaler()
scaled_x_data = scaler_x.fit_transform(x_data)
scaled_y_data = scaler_y.fit_transform(y_data)
scaled_y_data
np.column_stack([scaled_x_data, scaled_y_data])
df[['지역명 2s', '연도s', '월s']] = scaled_x_data
df['평당분양가격s'] = scaled_y_data
df.head()

,지역명,평당분양가격,연도,월,지역명2,지역명 2n,연도n,월n,평당분양가격n,지역명 2s,연도s,월s,평당분양가격s
0,서울,0.328198,0.0,1.0,8,0.5000,0.0,1.0,1.168591,0.000000,-1.875367,1.62196,1.168591
1,부산,0.065274,0.0,1.0,7,0.4375,0.0,1.0,-0.728312,-0.204124,-1.875367,1.62196,-0.728312
2,대구,0.064466,0.0,1.0,5,0.3125,0.0,1.0,-0.734147,-0.612372,-1.875367,1.62196,-0.734147
3,인천,0.119878,0.0,1.0,11,0.6875,0.0,1.0,-0.334363,0.612372,-1.875367,1.62196,-0.334363
4,광주,0.012757,0.0,1.0,4,0.2500,0.0,1.0,-1.107203,-0.816497,-1.875367,1.62196,-1.107203


# 5. 지역명을 원핫인코딩

In [28]:
#원핫 인코딩
from tensorflow.keras.utils import to_categorical
one_hot_encoded_data = to_categorical(labeled_data)
one_hot_encoded_data

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.]], dtype=float32)

In [38]:
#지역명, 지역명2
loc_info = df[['지역명', '지역명2']].head(17).sort_values(by='지역명2')
loc_column_names = loc_info['지역명'].tolist()
print(loc_column_names)

['강원', '경기', '경남', '경북', '광주', '대구', '대전', '부산', '서울', '세종', '울산', '인천', '전남', '전북', '제주', '충남', '충북']


In [43]:
df[loc_column_names] = to_categorical(df['지역명2']).astype(np.int16)
df.head()

,지역명,평당분양가격,연도,월,지역명2,지역명 2n,연도n,월n,평당분양가격n,지역명 2s,...,부산,서울,세종,울산,인천,전남,전북,제주,충남,충북
0,서울,0.328198,0.0,1.0,8,0.5000,0.0,1.0,1.168591,0.000000,...,0,1,0,0,0,0,0,0,0,0
1,부산,0.065274,0.0,1.0,7,0.4375,0.0,1.0,-0.728312,-0.204124,...,1,0,0,0,0,0,0,0,0,0
2,대구,0.064466,0.0,1.0,5,0.3125,0.0,1.0,-0.734147,-0.612372,...,0,0,0,0,0,0,0,0,0,0
3,인천,0.119878,0.0,1.0,11,0.6875,0.0,1.0,-0.334363,0.612372,...,0,0,0,0,1,0,0,0,0,0
4,광주,0.012757,0.0,1.0,4,0.2500,0.0,1.0,-1.107203,-0.816497,...,0,0,0,0,0,0,0,0,0,0


In [44]:
pd.options.display.max_columns # 최대 출력가능한 데이터프레임 열수
df.head()

,지역명,평당분양가격,연도,월,지역명2,지역명 2n,연도n,월n,평당분양가격n,지역명 2s,...,부산,서울,세종,울산,인천,전남,전북,제주,충남,충북
0,서울,0.328198,0.0,1.0,8,0.5000,0.0,1.0,1.168591,0.000000,...,0,1,0,0,0,0,0,0,0,0
1,부산,0.065274,0.0,1.0,7,0.4375,0.0,1.0,-0.728312,-0.204124,...,1,0,0,0,0,0,0,0,0,0
2,대구,0.064466,0.0,1.0,5,0.3125,0.0,1.0,-0.734147,-0.612372,...,0,0,0,0,0,0,0,0,0,0
3,인천,0.119878,0.0,1.0,11,0.6875,0.0,1.0,-0.334363,0.612372,...,0,0,0,0,1,0,0,0,0,0
4,광주,0.012757,0.0,1.0,4,0.2500,0.0,1.0,-1.107203,-0.816497,...,0,0,0,0,0,0,0,0,0,0
